In [1]:
# !pip -q install pypdf openai chromadb


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
from openai import OpenAI

# Point the client to your local Ollama server
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

# Pdf reader

In [7]:
import os
from pypdf import PdfReader

folder_path = r"D:\python_projects\zends_project\local\knowledge_base"

pdf_files = [f for f in os.listdir(folder_path) if f.lower().endswith('.pdf')]
print(f'Found {len(pdf_files)} PDF file(s): {pdf_files}')

all_documents = []
all_texts = []  # Accumulator for all files

for filename in pdf_files:
    pdf_path = os.path.join(folder_path, filename)
    print(f'\nProcessing: {filename}')

    reader = PdfReader(pdf_path)
    pages = []

    for page_num, page in enumerate(reader.pages, start=1):
        page_text = page.extract_text() or ''
        pages.append({
            'page_number': page_num,
            'text': page_text
        })

    file_text = '\n'.join(page['text'] for page in pages)
    all_texts.append(file_text)  # Store text per file

    all_documents.append({
        'file_name': filename,
        'total_pages': len(pages),
        'pages': pages,
        'text': file_text
    })

    print(f'  - Total pages: {len(pages)}')
    print(f'  - Total text characters: {len(file_text)}')

# Combine text from ALL PDF files together
text = '\n'.join(all_texts)
print(f'\nTotal combined text characters across all files: {len(text)}')

Found 2 PDF file(s): ['company_profile.pdf', 'telecom_knowledge_base.pdf']

Processing: company_profile.pdf
  - Total pages: 4
  - Total text characters: 9831

Processing: telecom_knowledge_base.pdf
  - Total pages: 4
  - Total text characters: 4617

Total combined text characters across all files: 14449


# Chunking

In [8]:
chunk_size = 500
overlap = 50

chunks = []
start = 0
chunk_id = 0

while start < len(text):
    end = start + chunk_size
    chunk_text = text[start:end].strip()

    if chunk_text:
        chunks.append({
            'id': f"chunk-{chunk_id}",
            'chunk_text': chunk_text
        })
        chunk_id += 1

    start += (chunk_size - overlap)

print(f'Total chunks: {len(chunks)}')
print('\nFirst chunk:\n')
print(chunks[0]['chunk_text'])


Total chunks: 33

First chunk:

ZENDS Communications
Powering Digital Life Everywhere
Company Overview
ZENDS Communications is a next-generation global telecom and digital services provider delivering mobile
connectivity,  broadband,  enterprise  networking,  cloud  infrastructure,  IoT,  cybersecurity,  and  digital
platforms. ZENDS serves both individual consumers and enterprise customers across multiple regions,
enabling personal communication, business operations, cloud computing, and smart-city services. ZENDS
operates ac


# generate embeddings

In [9]:
# 1. Prepare your chunk texts
chunk_texts = [chunk['chunk_text'] for chunk in chunks]

# 2. Define the embedding function using Ollama
def get_embeddings(texts, model="mxbai-embed-large", batch_size=100):
    if isinstance(texts, str):
        texts = [texts]
        
    all_embeddings = []
    
    # Process in batches to handle large lists of chunks smoothly
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        
        response = client.embeddings.create(
            model=model,
            input=batch
        )
        
        batch_embeddings = [item.embedding for item in response.data]
        all_embeddings.extend(batch_embeddings)
        
    return all_embeddings

# 3. Generate the chunk embeddings using Ollama
print("Generating embeddings via Ollama...")
chunk_embeddings = get_embeddings(chunk_texts, batch_size=100)

# 4. Print verification metrics
print('no. of embeddings:', len(chunk_embeddings))
print('embedding dimensions:', len(chunk_embeddings[0]))  # Should be 1024 for mxbai-embed-large
print('first 5 values:', chunk_embeddings[0][:5])

Generating embeddings via Ollama...
no. of embeddings: 33
embedding dimensions: 1024
first 5 values: [-0.06657034158706665, -0.016844185069203377, -0.019197972491383553, 0.03292934224009514, 0.01145114190876484]


# Vector DB

In [10]:
import chromadb

chromadb_client = chromadb.PersistentClient(path=  r"D:\python_projects\zends_project\local\vector_db\chroma_db" )
                                                    
collection = chromadb_client.get_or_create_collection(name="mannual")

existing = collection.get()
if existing['ids']:
    collection.delete(ids=existing['ids'])

collection.add(
    ids=[chunk['id'] for chunk in chunks],
    documents=[chunk['chunk_text'] for chunk in chunks],
    embeddings=chunk_embeddings,
    metadatas=[{'source': 'pdf_document'} for chunk in chunks]
)

print('Chunks stored in vector DB:', collection.count())

Chunks stored in vector DB: 33


# User query embedding

In [15]:
question ='may i know the refund rules'

question_embedding = get_embeddings([question])[0]

print("questions:", question)
print("question_embedding dimension", len(question_embedding))

questions: may i know the refund rules
question_embedding dimension 1024


# Retriever

In [16]:
results = collection.query(query_embeddings=[question_embedding], n_results=3)

retreived_chunks = results['documents'][0]
retreived_ids = results['ids'][0]
distance = results['distances'][0]

print('retreived_chunks: \n')

for i,(chunk_id,chunk_text,distance) in enumerate(zip(retreived_ids,retreived_chunks,distance),start = 1):
  print(f' result{i} | {chunk_id} | distance = {distance:.4f}')
  print(chunk_text[:500])
  print()

retreived_chunks: 

 result1 | chunk-28 | distance = 0.7652
an expected?
Possible reasons: Extra data usage, international roaming charges, or add-on services.
Q2: When is my payment due?

Bills must be paid within 7 days of invoice generation. Late payment may lead to temporary service
suspension.
Q3: Can I get a refund?
Refund policy: Full refund within 7 days if usage is below 10%. Cloud services are non-refundable
after activation.
7. Customer Support – FAQs
ZENDS provides 24×7 support services.
Q1: How can I contact customer support?
Customers can r

 result2 | chunk-20 | distance = 0.7792
maintenance.
Company Policies
Billing: Monthly billing in advance. Enterprise customers receive consolidated invoices. Late payment after 7
days may suspend services.
Refund:  Full  refund  within  7  days  if  usage  is  less  than  10%.  Cloud  services  are  non-refundable  after
activation.
3
Contracts: Individual users have no long-term lock-in. Enterprise customers have a minimum 12-month


# context + prompt

In [17]:
context = '\n\n---- Retrieved Chunk ----\n\n'.join(retreived_chunks)

prompt = f"""
Answer the user's question using ONLY the retrieved Context below.
If the answer is not present in the context, say that the information is not available in the provided document.
Retrieved Context :{context}
User Question:{question}""".strip()

print(prompt)



Answer the user's question using ONLY the retrieved Context below.
If the answer is not present in the context, say that the information is not available in the provided document.
Retrieved Context :an expected?
Possible reasons: Extra data usage, international roaming charges, or add-on services.
Q2: When is my payment due?

Bills must be paid within 7 days of invoice generation. Late payment may lead to temporary service
suspension.
Q3: Can I get a refund?
Refund policy: Full refund within 7 days if usage is below 10%. Cloud services are non-refundable
after activation.
7. Customer Support – FAQs
ZENDS provides 24×7 support services.
Q1: How can I contact customer support?
Customers can r

---- Retrieved Chunk ----

maintenance.
Company Policies
Billing: Monthly billing in advance. Enterprise customers receive consolidated invoices. Late payment after 7
days may suspend services.
Refund:  Full  refund  within  7  days  if  usage  is  less  than  10%.  Cloud  services  are  non-refund

# generate answer

In [18]:
response = client.chat.completions.create(
    model="llama3.2:1b",
    temperature=0.1,  # Keep temperature low for exact adherence
    messages=[
        {
            "role": "system", 
            "content": "You are a precise telecom support assistant. You must always extract and output both the causes and the solutions when present in the context."
        },
        {
            "role": "user", 
            "content": prompt
        }
    ]
)

answer = response.choices[0].message.content
print(answer)

Based on the retrieved Context, the refund rules are as follows:

* For individual users, a full refund is available within 7 days if usage is below 10%.
* For enterprise customers, cloud services are non-refundable after activation, and full refunds are available within 7 days if usage is below 10%.
* There is no refund policy mentioned for ZENDS Cloud services.

Note that the refund policy for ZENDS Cloud services is non-refundable after activation, but the refund policy for individual users and enterprise customers is different.


In [21]:
import chromadb
from openai import OpenAI

client = chromadb.PersistentClient(path=r"D:\python_projects\zends_project\local\vector_db\chroma_db")
col = client.get_collection("mannual")

print("Total chunks:", col.count())

sample = col.peek(limit=1)
emb_list = sample.get("embeddings")
if emb_list is not None and len(emb_list) > 0:
    print("Stored embedding dimension:", len(emb_list[0]))
else:
    print("Stored embedding dimension: None / not returned")

ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
resp = ollama.embeddings.create(model="mxbai-embed-large", input=["wifi disconnection issue"])
query_vec = resp.data[0].embedding
print("Ollama query embedding dimension:", len(query_vec))

res = col.query(query_embeddings=[query_vec], n_results=3)
docs = res.get("documents", [[]])[0]
print("Matched chunks returned:", len(docs))
for i, doc in enumerate(docs, 1):
    print(f"\n--- Chunk {i} ---")
    print(doc[:250])

Total chunks: 33
Stored embedding dimension: 1024
Ollama query embedding dimension: 1024
Matched chunks returned: 3

--- Chunk 1 ---
ossible causes: Network congestion, router issues, or high number of connected devices.
Solution: Restart the router and check connected devices.
Q2: My internet is completely down. What should I do?
Possible causes: Fiber cable damage, service outag

--- Chunk 2 ---
router and WiFi equipment, installation, 24×7 technical support, network
monitoring, setup guidance, troubleshooting, optional bundled plans with mobile connectivity, and SLA-
backed support for enterprise customers.
3. Business Connectivity
ZENDBiz 

--- Chunk 3 ---
data settings or restart your device.
Q3: Why are my calls dropping frequently?
Possible causes: Network congestion, weak signal, or device compatibility issues.
Solution: Switch to 4G/5G mode or move to a stronger signal location.
5. Broadband Servi
